# DENUE Morelia: búsqueda exploratoria de correspondencias 2013–2015

**Subsector SCIAN 722: servicios de preparación de alimentos y bebidas.**

Esta versión se preparó a partir de la bitácora original de 132 celdas, realizada entre el 6 y el 7 de octubre de 2026. Conserva el procedimiento **exploratorio** de preparación, generación y diagnóstico de candidatos, pero **no replica como resultados definitivos las decisiones manuales** de la bitácora.

**Muy importante:** un candidato no es un enlace confirmado. Un enlace tampoco demuestra, por sí solo, supervivencia continua. En este notebook **no se estiman nacimientos, cierres ni supervivencia empresarial**, ni se aplica un modelo probabilístico calibrado ni una asignación final. Eso continúa pendiente de validación.

Las salidas se publican vacías para evitar divulgar por accidente nombres ligados a teléfonos y registros individuales. El notebook sólo imprime **resultados agregados**. Las bases originales y los expedientes de revisión se mantienen fuera del repositorio público.


## 1. Configuración y comprobación de archivos

Trabaja con los dos archivos `DENUE_2013_MORELIA.csv` y `DENUE_2015_MORELIA.csv`, elaborados previamente desde los cortes históricos. Para evitar mezclar versiones, se verifican los tamaños y las huellas SHA256 obtenidas durante la auditoría local. Si tus archivos son distintos, primero hay que explicar por qué; **no desactives los controles sólo para hacer que corra**.

Puedes indicar la carpeta de archivos mediante la variable de entorno `DENUE_MORELIA_DIR`. Si no la configuras, el notebook la buscará en las ubicaciones habituales del proyecto o en la carpeta hermana `DENUE_HISTORICOS`.


In [ ]:
from pathlib import Path
import os
import re
import hashlib
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

FUENTES = {
    2013: {
        "archivo": "DENUE_2013_MORELIA.csv",
        "filas_morelia": 42672,
        "filas_722": 4239,
        "id": "Llave DENUE",
        "sha256": "2d597d24038e48163a0dd1446d9d34e20d7eb0e936f1d41736cb508716ca1aa9",
    },
    2015: {
        "archivo": "DENUE_2015_MORELIA.csv",
        "filas_morelia": 44814,
        "filas_722": 5376,
        "id": "ID",
        "sha256": "ad2dc4910aeb54f0c8b7d66da4bfc56f0b563fabeac618213791c9019941bbce",
    },
}


def localizar_fuentes():
    entorno = os.environ.get("DENUE_MORELIA_DIR", "").strip()
    cwd = Path.cwd().resolve()
    opciones = ([Path(entorno)] if entorno else []) + [
        cwd / "01_MORELIA_LIMPIO",
        cwd.parent / "01_MORELIA_LIMPIO",
        cwd.parent.parent / "01_MORELIA_LIMPIO",
        cwd.parent.parent / "DENUE_HISTORICOS" / "01_MORELIA_LIMPIO",
    ]
    for directorio in opciones:
        if all((directorio / f["archivo"]).is_file() for f in FUENTES.values()):
            return directorio.resolve()
    raise FileNotFoundError(
        "No encuentro los dos CSV originales de Morelia. "
        "Configura DENUE_MORELIA_DIR con la carpeta 01_MORELIA_LIMPIO."
    )

DATA_DIR = localizar_fuentes()


def leer_verificado(anio):
    ficha = FUENTES[anio]
    ruta = DATA_DIR / ficha["archivo"]
    with ruta.open("rb") as archivo:
        huella = hashlib.file_digest(archivo, "sha256").hexdigest()
    assert huella == ficha["sha256"], (
        f"El CSV {anio} no coincide con la fuente auditada (SHA256 distinto)."
    )
    tabla = pd.read_csv(ruta, dtype=str, low_memory=False)
    assert len(tabla) == ficha["filas_morelia"], f"Filas inesperadas en {anio}"
    assert tabla.index.is_unique
    assert tabla[ficha["id"]].notna().all()
    assert tabla[ficha["id"]].is_unique
    return tabla

fuentes_originales = {anio: leer_verificado(anio) for anio in FUENTES}
print("Maestros originales verificados por SHA256 y número de filas.")
for anio, df in fuentes_originales.items():
    print(f"{anio}: {len(df):,} registros de Morelia (denominador: todo Morelia).")


## 2. Preparación comparable sin alterar las fuentes

Conservamos los identificadores originales del DENUE, los índices de fila del CSV, la clasificación SCIAN y los valores originales. Las columnas normalizadas se construyen por separado. La diferencia entre la «Llave DENUE» de 2013 y el «ID» de 2015 **no se convierte en correspondencia**.

Los valores faltantes no se consideran acuerdos. Las coordenadas pueden ser formalmente válidas y, aun así, estar repetidas o desplazadas.


In [ ]:
VACIOS = {
    "", "0", "NINGUNO", "SIN DATO", "SIN NUMERO", "S N", "SN",
    "NO DISPONIBLE", "NO ESPECIFICADO", "SIN INFORMACION", "N A",
}


def texto_normalizado(serie):
    return (
        serie.fillna("").astype(str)
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
        .str.upper()
        .str.replace(r"[^A-Z0-9]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def telefono_normalizado(serie):
    digitos = serie.fillna("").astype(str).str.replace(r"\D", "", regex=True)
    valido = digitos.map(lambda s: 7 <= len(s) <= 13 and len(set(s)) > 1)
    return digitos.where(valido, "")


def nombre_sin_sufijo(serie):
    return serie.str.replace(r"\s+SIN NOMBRE$", "", regex=True).str.strip()


def preparar_maestro(anio, df):
    requerido = {
        "Nombre de la Unidad Económica", "Razón social", "Código de la clase de actividad SCIAN",
        "Nombre de la vialidad", "Número exterior o kilómetro",
        "Nombre de asentamiento humano", "Latitud", "Longitud",
        "Número de teléfono" if anio == 2015 else "Número de teléfono 1",
    }
    if anio == 2013:
        requerido.add("Número de teléfono 2")
    assert requerido.issubset(df.columns), f"Columnas ausentes en {anio}: {requerido - set(df.columns)}"

    m = pd.DataFrame(index=df.index.copy())
    m["id_fila"] = df.index.astype(int)
    m["denue_id_original"] = df[FUENTES[anio]["id"]]
    m["scian_original"] = df["Código de la clase de actividad SCIAN"].fillna("").str.strip()
    m["es_722"] = m["scian_original"].str.startswith("722", na=False)
    m["nombre_original"] = df["Nombre de la Unidad Económica"]
    m["nombre_norm"] = texto_normalizado(m["nombre_original"])
    m["nombre_base"] = nombre_sin_sufijo(m["nombre_norm"])
    m["razon_norm"] = texto_normalizado(df["Razón social"])
    for campo, col in {
        "vialidad": "Nombre de la vialidad",
        "numero": "Número exterior o kilómetro",
        "colonia": "Nombre de asentamiento humano",
    }.items():
        m[f"{campo}_norm"] = texto_normalizado(df[col])
    t1 = "Número de teléfono" if anio == 2015 else "Número de teléfono 1"
    m["telefono_1"] = telefono_normalizado(df[t1])
    m["telefono_2"] = (telefono_normalizado(df["Número de teléfono 2"])
                       if anio == 2013 else "")
    m["lat"] = pd.to_numeric(df["Latitud"], errors="coerce")
    m["lon"] = pd.to_numeric(df["Longitud"], errors="coerce")
    m["geo_valida"] = (
        m["lat"].between(-90, 90) & m["lon"].between(-180, 180)
        & m["lat"].ne(0) & m["lon"].ne(0)
    )
    assert m.index.is_unique
    assert m["id_fila"].is_unique
    assert m["denue_id_original"].is_unique
    assert int(m["es_722"].sum()) == FUENTES[anio]["filas_722"]
    return m

maestros = {anio: preparar_maestro(anio, tabla) for anio, tabla in fuentes_originales.items()}
u13 = maestros[2013].loc[maestros[2013]["es_722"]].copy()
u15 = maestros[2015].loc[maestros[2015]["es_722"]].copy()
assert len(u13) == 4239 and len(u15) == 5376
print("Universo 722 comprobado: 2013: 4,239 establecimientos; 2015: 5,376 establecimientos.")


## 3. Disponibilidad de variables (denominador: establecimientos de cada corte)

Tener un nombre registrado no significa que sea identificador distintivo: muchos nombres son genéricos. Tampoco confundimos ausencia de teléfono con un teléfono distinto.


In [ ]:
def disponibilidad_maestro(u):
    resultado = {
        "nombre": ~u["nombre_norm"].isin(VACIOS),
        "razon_social": ~u["razon_norm"].isin(VACIOS),
        "vialidad": ~u["vialidad_norm"].isin(VACIOS),
        "numero": ~u["numero_norm"].isin(VACIOS),
        "colonia": ~u["colonia_norm"].isin(VACIOS),
        "telefono": u["telefono_1"].ne("") | u["telefono_2"].ne(""),
        "coordenadas": u["geo_valida"],
    }
    return pd.DataFrame(resultado, index=u.index)

presencia = {2013: disponibilidad_maestro(u13), 2015: disponibilidad_maestro(u15)}
for anio in (2013, 2015):
    tabla = presencia[anio]
    print(f"\n{anio} | denominador: {len(tabla):,} establecimientos")
    for campo in tabla.columns:
        n = int(tabla[campo].sum())
        print(f"{campo}: {n:,} establecimientos ({100*n/len(tabla):.2f}%)")


## 4. Generación de candidatos: cinco rutas, sin aceptar enlaces

Las cinco rutas recuperan posibles parejas: nombre exacto, nombre sin el sufijo «SIN NOMBRE», teléfono, calle+número y proximidad de **250 metros**. Este radio sirve **sólo para ampliar la búsqueda**. No prueba identidad ni significa que un establecimiento haya cambiado de ubicación. Las rutas se unen sin descartar candidatos competidores.

La comparación en esta fase se limita a establecimientos 722; una búsqueda auxiliar en otras clases SCIAN está prevista y aún no está realizada.


In [ ]:
R_TIERRA = 6371000.0
R_BUSQUEDA_M = 250.0


def tabla_telefonos(u, columna_id):
    partes = []
    for col in ("telefono_1", "telefono_2"):
        t = u[[columna_id, col]].rename(columns={col: "telefono"}).copy()
        partes.append(t.loc[t["telefono"].ne("")])
    return pd.concat(partes, ignore_index=True).drop_duplicates([columna_id, "telefono"])


def xyz_metros(u):
    lat = np.radians(u["lat"].to_numpy(dtype=float))
    lon = np.radians(u["lon"].to_numpy(dtype=float))
    return R_TIERRA * np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


def generar_candidatos(a, b):
    a = a.copy()
    b = b.copy()
    a["id_2013"] = a.index.to_numpy(dtype=int)
    b["id_2015"] = b.index.to_numpy(dtype=int)
    rutas = []

    def agregar(pares, nombre_ruta):
        pares = pares[["id_2013", "id_2015"]].drop_duplicates().copy()
        pares["ruta"] = nombre_ruta
        rutas.append(pares)
        print(f"{nombre_ruta}: {len(pares):,} pares candidatos")

    invalidos_nombre = {"", "SIN NOMBRE"}
    na = a.loc[~a["nombre_norm"].isin(invalidos_nombre), ["id_2013", "nombre_norm"]]
    nb = b.loc[~b["nombre_norm"].isin(invalidos_nombre), ["id_2015", "nombre_norm"]]
    agregar(na.merge(nb, on="nombre_norm"), "NOMBRE_EXACTO")

    na = a.loc[~a["nombre_base"].isin(invalidos_nombre), ["id_2013", "nombre_base"]]
    nb = b.loc[~b["nombre_base"].isin(invalidos_nombre), ["id_2015", "nombre_base"]]
    agregar(na.merge(nb, on="nombre_base"), "NOMBRE_BASE")

    ta = tabla_telefonos(a, "id_2013")
    tb = tabla_telefonos(b, "id_2015")
    agregar(ta.merge(tb, on="telefono"), "TELEFONO")

    da = a.loc[(~a["vialidad_norm"].isin(VACIOS)) & (~a["numero_norm"].isin(VACIOS)),
               ["id_2013", "vialidad_norm", "numero_norm"]]
    db = b.loc[(~b["vialidad_norm"].isin(VACIOS)) & (~b["numero_norm"].isin(VACIOS)),
               ["id_2015", "vialidad_norm", "numero_norm"]]
    agregar(da.merge(db, on=["vialidad_norm", "numero_norm"]), "DOMICILIO")

    ga = a.loc[a["geo_valida"]].copy()
    gb = b.loc[b["geo_valida"]].copy()
    arbol = cKDTree(xyz_metros(gb))
    radio_cuerda = 2 * R_TIERRA * np.sin(R_BUSQUEDA_M / (2 * R_TIERRA))
    vecinos = arbol.query_ball_point(xyz_metros(ga), r=radio_cuerda)
    conteos = np.array([len(vecinos_fila) for vecinos_fila in vecinos], dtype=int)
    if conteos.sum():
        indices_b = np.concatenate([np.asarray(v, dtype=int) for v in vecinos if len(v)])
        geo = pd.DataFrame({
            "id_2013": np.repeat(ga["id_2013"].to_numpy(dtype=int), conteos),
            "id_2015": gb["id_2015"].to_numpy(dtype=int)[indices_b],
        })
    else:
        geo = pd.DataFrame(columns=["id_2013", "id_2015"])
    agregar(geo, "GEOGRAFIA_250M")

    todos = pd.concat(rutas, ignore_index=True).drop_duplicates()
    unidos = todos.groupby(["id_2013", "id_2015"], as_index=False).agg(
        rutas=("ruta", lambda s: "|".join(sorted(set(s))) )
    )
    assert not unidos.duplicated(["id_2013", "id_2015"]).any()
    assert unidos["id_2013"].isin(a.index).all()
    assert unidos["id_2015"].isin(b.index).all()
    return unidos

candidatos = generar_candidatos(u13, u15)
assert len(candidatos) == 162512, "No se reprodujo el conteo auditado de candidatos."
print(f"\nPares únicos candidatos: {len(candidatos):,}")
print(f"2013 con candidatos: {candidatos['id_2013'].nunique():,} de 4,239 establecimientos")
print(f"2015 con candidatos: {candidatos['id_2015'].nunique():,} de 5,376 establecimientos")


## 5. Diagnóstico: por qué no todas las rutas sirven para aceptar enlaces

Un gran número de pares se recuperó por cercanía o por nombres genéricos. Para evitar confundir cantidad con calidad, distinguimos **rutas** y **familias de evidencia**; nombre exacto y nombre base son una misma familia. Calle y geografía están relacionadas y tampoco deben sumarse como evidencias independientes en un modelo probabilístico sin comprobar esa dependencia.


In [ ]:
banderas = candidatos["rutas"].str.get_dummies(sep="|")
rutas_esperadas = ["NOMBRE_EXACTO", "NOMBRE_BASE", "TELEFONO", "DOMICILIO", "GEOGRAFIA_250M"]
assert set(banderas.columns) == set(rutas_esperadas)
banderas = banderas[rutas_esperadas]

familias = pd.DataFrame({
    "NOMBRE": banderas["NOMBRE_EXACTO"].eq(1) | banderas["NOMBRE_BASE"].eq(1),
    "TELEFONO": banderas["TELEFONO"].eq(1),
    "DOMICILIO": banderas["DOMICILIO"].eq(1),
    "GEOGRAFIA": banderas["GEOGRAFIA_250M"].eq(1),
})
numero_familias = familias.sum(axis=1)
print("Pares por número de familias de recuperación (denominador: 162,512 pares):")
print(numero_familias.value_counts().sort_index().to_string())
print("Sólo nombre:", int((familias["NOMBRE"] & numero_familias.eq(1)).sum()))
print("Sólo geografía:", int((familias["GEOGRAFIA"] & numero_familias.eq(1)).sum()))

f13 = u13["nombre_base"].value_counts()
f15 = u15["nombre_base"].value_counts()
frecuentes = f13.rename("n2013").to_frame().join(f15.rename("n2015"), how="inner")
frecuentes = frecuentes.loc[~frecuentes.index.isin(["", "SIN NOMBRE"])].copy()
frecuentes["posibles"] = frecuentes["n2013"] * frecuentes["n2015"]
assert int(frecuentes["posibles"].sum()) == int(banderas["NOMBRE_BASE"].sum())
print("Mayor bloque genérico por nombre base (número de pares posibles):", int(frecuentes["posibles"].max()))


## 6. Comparaciones: acuerdo, desacuerdo y falta de información

Un teléfono ausente no es «teléfono distinto». Para nombre, razón social, calle, número, colonia y teléfono se distinguen tres estados. Los acuerdos no equivalen a identidades validadas. Distancia geográfica y similitud textual permanecen como medidas **continuas**, sin umbral de aceptación automática.


In [ ]:
def estados_texto(x, y):
    x = x.reset_index(drop=True)
    y = y.reset_index(drop=True)
    ambos = ~x.isin(VACIOS) & ~y.isin(VACIOS)
    return np.where(~ambos, "NO_COMPARABLE", np.where(x.eq(y), "ACUERDO", "DESACUERDO"))


def haversine_m(a, b):
    lat1, lat2 = np.radians(a["lat"].to_numpy(float)), np.radians(b["lat"].to_numpy(float))
    lon1, lon2 = np.radians(a["lon"].to_numpy(float)), np.radians(b["lon"].to_numpy(float))
    h = (np.sin((lat2 - lat1)/2)**2 +
         np.cos(lat1)*np.cos(lat2)*np.sin((lon2 - lon1)/2)**2)
    valido = a["geo_valida"].to_numpy(bool) & b["geo_valida"].to_numpy(bool)
    return np.where(valido, 2*R_TIERRA*np.arcsin(np.sqrt(np.clip(h, 0, 1))), np.nan)


def comparar_pares(c, a, b):
    x = a.loc[c["id_2013"].to_numpy()].reset_index(drop=True)
    y = b.loc[c["id_2015"].to_numpy()].reset_index(drop=True)
    salida = c.copy().reset_index(drop=True)
    for campo in ("nombre_base", "razon_norm", "vialidad_norm", "numero_norm", "colonia_norm"):
        salida[f"estado_{campo}"] = estados_texto(x[campo], y[campo])
    # Calle y número: evidencia conjunta; no sumarla de nuevo sin modelar dependencia.
    vx, vy = x["vialidad_norm"], y["vialidad_norm"]
    nx, ny = x["numero_norm"], y["numero_norm"]
    ambas = ~vx.isin(VACIOS) & ~vy.isin(VACIOS) & ~nx.isin(VACIOS) & ~ny.isin(VACIOS)
    salida["estado_calle_numero"] = np.where(
        ~ambas, "NO_COMPARABLE",
        np.where(vx.eq(vy) & nx.eq(ny), "ACUERDO", "DESACUERDO")
    )
    tx1, tx2 = x["telefono_1"], x["telefono_2"]
    ty1, ty2 = y["telefono_1"], y["telefono_2"]
    tel_x = tx1.ne("") | tx2.ne("")
    tel_y = ty1.ne("") | ty2.ne("")
    tel_ok = ((tx1.ne("") & (tx1.eq(ty1) | tx1.eq(ty2))) |
              (tx2.ne("") & (tx2.eq(ty1) | tx2.eq(ty2))))
    salida["estado_telefono"] = np.where(
        ~(tel_x & tel_y), "NO_COMPARABLE", np.where(tel_ok, "ACUERDO", "DESACUERDO")
    )
    f13 = a["nombre_base"].value_counts()
    f15 = b["nombre_base"].value_counts()
    salida["frecuencia_nombre_2013"] = x["nombre_base"].map(f13).to_numpy(dtype=int)
    salida["frecuencia_nombre_2015"] = y["nombre_base"].map(f15).to_numpy(dtype=int)
    salida["nombre_frecuente_10_diagnostico"] = (
        salida["frecuencia_nombre_2013"].ge(10) | salida["frecuencia_nombre_2015"].ge(10)
    )
    salida["similitud_nombre"] = np.fromiter(
        (100*SequenceMatcher(None, s, t, autojunk=False).ratio() if s and t else 0
         for s,t in zip(x["nombre_base"], y["nombre_base"])),
        dtype=float, count=len(c)
    )
    salida["distancia_m"] = haversine_m(x, y)
    salida["mismo_scian_original"] = x["scian_original"].eq(y["scian_original"]).to_numpy()
    assert len(salida) == len(c)
    assert not salida.duplicated(["id_2013", "id_2015"]).any()
    assert salida["similitud_nombre"].between(0, 100).all()
    return salida

comparaciones = comparar_pares(candidatos, u13, u15)
print(f"Pares comparados: {len(comparaciones):,} (denominador: pares candidatos)")
for campo in ["nombre_base", "razon_norm", "vialidad_norm", "numero_norm", "colonia_norm", "calle_numero", "telefono"]:
    k = f"estado_{campo}"
    conteos = comparaciones[k].value_counts()
    print(f"{campo}: acuerdo={conteos.get('ACUERDO',0):,}, desacuerdo={conteos.get('DESACUERDO',0):,}, no comparable={conteos.get('NO_COMPARABLE',0):,}")
print("Distancia mediana de los candidatos (m):", round(float(comparaciones["distancia_m"].median()), 2))
print("Nombres con similitud >=80 (sólo diagnóstico):", int(comparaciones["similitud_nombre"].ge(80).sum()))


## 7. ¿Qué tan identificadores son los teléfonos coincidentes?

Medimos la frecuencia de cada teléfono en **todo Morelia**, no sólo entre registros pendientes o dentro del subsector 722. Un teléfono «exclusivo» en los archivos aporta evidencia potencialmente útil, pero **no garantiza continuidad** ni descarta la reasignación del número. No publicamos los números individuales.


In [ ]:
def telefonos_morelia(maestro, anio):
    tabla = maestro[["id_fila", "telefono_1", "telefono_2"]]
    trozos = []
    for campo in ("telefono_1", "telefono_2"):
        t = tabla[["id_fila", campo]].rename(columns={campo: "telefono"}).copy()
        trozos.append(t.loc[t["telefono"].ne("")])
    return pd.concat(trozos, ignore_index=True).drop_duplicates(["id_fila", "telefono"])

telefonos_totales = {anio: telefonos_morelia(maestros[anio], anio) for anio in (2013, 2015)}
frecuencias = {anio: t.groupby("telefono")["id_fila"].nunique() for anio, t in telefonos_totales.items()}
t13 = telefonos_totales[2013].loc[
    telefonos_totales[2013]["id_fila"].isin(u13.index)
].rename(columns={"id_fila":"id_2013"})
t15 = telefonos_totales[2015].loc[
    telefonos_totales[2015]["id_fila"].isin(u15.index)
].rename(columns={"id_fila":"id_2015"})
pares_telefono = t13.merge(t15, on="telefono")
pares_telefono["exclusivo_morelia"] = (
    pares_telefono["telefono"].map(frecuencias[2013]).eq(1)
    & pares_telefono["telefono"].map(frecuencias[2015]).eq(1)
)
telefono_pares = pares_telefono.groupby(["id_2013", "id_2015"], as_index=False).agg(
    exclusivo_morelia=("exclusivo_morelia", "max")
)
assert len(telefono_pares) == 461
assert len(telefono_pares) == int(comparaciones["estado_telefono"].eq("ACUERDO").sum())
assert int(telefono_pares["exclusivo_morelia"].sum()) == 418
print("Pares telefónicos coincidentes: 461; de ellos 418 con número exclusivo en ambos cortes de todo Morelia.")
print("Pares coincidentes con número compartido en al menos un corte: 43.")


## 8. Comparación opcional con los 1,146 enlaces anteriores

El archivo `ENLACES_PROVISIONALES_2013_2015_V1.csv` es una **bitácora local** de los enlaces aceptados antes de replantear el procedimiento. Contiene decisiones aún no validadas y **no se incluye en GitHub**. Si está disponible localmente, el notebook verificará únicamente cobertura y unicidad de sus identificadores; no los tratará como verdad de entrenamiento.

El notebook se puede ejecutar de principio a fin aunque ese archivo no esté disponible.


In [ ]:
def buscar_respaldo_local():
    entorno = os.environ.get("DENUE_ENLACES_PROVISIONALES", "").strip()
    cwd = Path.cwd().resolve()
    rutas = ([Path(entorno)] if entorno else []) + [
        cwd / "RESULTADOS_AUDITORIA" / "ENLACES_PROVISIONALES_2013_2015_V1.csv",
        cwd.parent / "06_DEMOGRAFIA_SERVICIOS_ALIMENTOS" / "RESULTADOS_AUDITORIA" / "ENLACES_PROVISIONALES_2013_2015_V1.csv",
        cwd.parent.parent / "DENUE_HISTORICOS" / "06_DEMOGRAFIA_SERVICIOS_ALIMENTOS"
        / "RESULTADOS_AUDITORIA" / "ENLACES_PROVISIONALES_2013_2015_V1.csv",
    ]
    return next((p for p in rutas if p.is_file()), None)

respaldo = buscar_respaldo_local()
if respaldo is None:
    print("No está disponible el respaldo privado V1; verificación opcional omitida.")
else:
    d = pd.read_csv(respaldo, usecols=["id_2013", "id_2015", "etapa", "decision_enlace"])
    assert len(d) == 1146
    assert d["id_2013"].is_unique and d["id_2015"].is_unique
    assert d["id_2013"].isin(u13.index).all() and d["id_2015"].isin(u15.index).all()
    cobertura = d.merge(candidatos[["id_2013", "id_2015"]],
                        on=["id_2013", "id_2015"], how="left", indicator=True,
                        validate="one_to_one")
    print("Enlaces provisionales en el respaldo local: 1,146 (812 respaldados, 334 probables inicialmente).")
    print("Recuperados por la nueva generación de candidatos:",
          int(cobertura["_merge"].eq("both").sum()), "de 1,146 enlaces provisionales")
    print("Nota: recuperación NO significa identidad validada.")


## 9. Lo que falta antes de hablar de trayectorias y supervivencia

- Completar la búsqueda de candidatos que cambian de nombre o clasificación, incluso **fuera de SCIAN 722**, manteniendo el universo analítico acotado.
- Medir el error de recuperación de candidatos mediante revisión que no dependa sólo de las rutas ya generadas.
- Construir una referencia de revisión manual: por ahora existe un **piloto de 55 expedientes y 1,427 pares candidatos**, no una validación estadística definitiva. La revisión la realizará una sola investigadora y se reconocerá esa limitación.
- Estimar y calibrar puntuaciones (por ejemplo, un esquema probabilístico con dependencias controladas). **No hay pesos finales estimados.**
- Aplicar asignación **uno-a-uno con opción de no enlazar**, y dejar los conflictos sin forzar.
- Evaluar sensibilidad, precisión y correspondencias omitidas con una muestra independiente, documentando denominadores e incertidumbre.
- Sólo después, construir trayectorias de **presencia observada** y, si los supuestos lo permiten, estudiar eventos demográficos. Aparecer no equivale automáticamente a nacer, desaparecer no equivale automáticamente a morir.

La nota explicativa para el profesor se encuentra en `DOCUMENTACION/NOTA_AVANCE_DENUE_2013_2015.md`.
